In [ ]:
%pip install pandas==2.3.3
%pip install numpy==2.3.3
%pip install matplotlib==3.10.7
%pip install scikit-learn==1.7.2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold, GridSearchCV, LeaveOneGroupOut
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer

In [ ]:
df_dog_moveset = pd.read_csv('../../data/processed/DogFeatures.csv')

print(df_dog_moveset.head())

cols_to_drop = ['label', 'DogID'] + [c for c in df_dog_moveset.columns if 'abs_sum' in c]
x = df_dog_moveset.drop(cols_to_drop, axis=1).values
y = df_dog_moveset['label'].values

classes = np.unique(y)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
params_gnb = {'var_smoothing': [1e-15, 1e-14, 1e-13, 1e-12, 1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]}

grid_search = GridSearchCV(GaussianNB(), param_grid=params_gnb, cv=5, verbose=1, scoring='accuracy')
grid_search.fit(x_train, y_train)

In [ ]:
# gnb = GaussianNB(var_smoothing=1e-15)

gnb = Pipeline([('scaler', StandardScaler()),
                ('power', PowerTransformer(method='yeo-johnson')),
                ('gnb', GaussianNB(var_smoothing=1e-15))])

In [ ]:
# Sem Cross-Validation

gnb.fit(x_train, y_train)

y_pred = gnb.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Acurácia do modelo: {accuracy * 100:.2f}%')

result_matrix = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix=result_matrix, display_labels=classes)
display.plot(xticks_rotation=45)

In [ ]:
# Com Cross-Validation

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(gnb, x, y, cv=kfold)
y_pred_cv = cross_val_predict(gnb, x, y, cv=kfold)

print(f"Acurácia para cada fold: {scores}")
print(f"Acurácia média: {scores.mean() * 100:.2f}%")

result_matrix = confusion_matrix(y, y_pred_cv)
display = ConfusionMatrixDisplay(confusion_matrix=result_matrix, display_labels=classes)
display.plot(xticks_rotation=45)

In [ ]:
dog_out = LeaveOneGroupOut()
dogs_id = (df_dog_moveset['DogID'].values)

In [ ]:
# Leave One Dog Out sem Cross-Validation

acc_scores = []
f1_weighted_scores = []
f1_macro_scores = []
dogs_tested = []

for train, test in dog_out.split(x, y, dogs_id):
    x_train, x_test = x[train], x[test]
    y_train, y_test = y[train], y[test]

    current_dog_id = dogs_id[test][0]

    gnb.fit(x_train, y_train)
    y_pred = gnb.predict(x_test)

    acc_scores.append(accuracy_score(y_test, y_pred))
    f1_weighted_scores.append(f1_score(y_test, y_pred, average='weighted'))
    f1_macro_scores.append(f1_score(y_test, y_pred, average='macro'))

    dogs_tested.append(current_dog_id)

print(f"Acurácia média: {np.mean(acc_scores) * 100:.2f}%")
print(f"F1-Score Ponderado médio: {np.mean(f1_weighted_scores) * 100:.2f}%")
print(f"F1-Score Macro médio: {np.mean(f1_macro_scores) * 100:.2f}%")

result_matrix = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix=result_matrix, display_labels=classes)
display.plot(xticks_rotation=45)

In [ ]:
# Leave One Dog Out com Cross-Validation

y_pred_cv = cross_val_predict(gnb, x, y, groups=dogs_id, cv=dog_out)
accuracy = accuracy_score(y, y_pred_cv)
print(f"Acurácia média: {accuracy * 100:.2f}%")

f1_weighted = f1_score(y, y_pred_cv, average='weighted')
f1_macro = f1_score(y, y_pred_cv, average='macro')
print(f"F1-Score Ponderado: {f1_weighted:.2%}")
print(f"F1-Score Macro: {f1_macro:.2%}")

result_matrix = confusion_matrix(y, y_pred_cv)
display = ConfusionMatrixDisplay(confusion_matrix=result_matrix, display_labels=classes)
display.plot(xticks_rotation=45)